In [8]:
from nnsight import LanguageModel

deepseek_model = LanguageModel("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
deepseek_tokenizer = deepseek_model.tokenizer


nemotron_model = LanguageModel("nvidia/Nemotron-Research-Reasoning-Qwen-1.5B")
nemotron_tokenizer = nemotron_model.tokenizer

In [19]:
import nnsight
@nnsight.trace
def streamer(tokens, model, max_length=80, state=None):
    # Initialize state if not provided
    if state is None:
        state = {'current_line': '', 'current_line_length': 0}

    token = tokens[-1] # only use last token

    # Decode the token
    decoded_token = model.tokenizer.decode(token).encode("unicode_escape").decode()

    if (decoded_token == '\\n') or (decoded_token == '\n'):  # Handle explicit newline tokens
        # Print the current line and reset state
        print('',flush=True)
        state['current_line'] = ''
        state['current_line_length'] = 0
    elif (decoded_token == '\n\n') or (decoded_token == '\\n\\n'):
        # Print the current line and reset state
        print('',flush=True)
        print('',flush=True)
        state['current_line'] = ''
        state['current_line_length'] = 0
    else:
        # Check if adding the token would exceed the max length
        if state['current_line_length'] + len(decoded_token) > max_length:
            print('',flush=True)
            state['current_line'] = decoded_token  # Start a new line with the current token
            state['current_line_length'] = len(decoded_token)
            print(decoded_token, flush=True, end="")  # Print ONLY the new token
        else:
            # Add a space if the line isn't empty and append the token
            if state['current_line']:
                state['current_line'] += decoded_token
            else:
                state['current_line'] = decoded_token
            state['current_line_length'] += len(decoded_token)
            print(decoded_token, flush=True, end="")  # Print ONLY the new token

    return state

/disk/u/gio/.conda/envs/retrieval/lib/python3.11/site-packages/nnsight/__init__.py:80: UserWarning: nnsight.trace is deprecated as of v0.5.0 and will be removed in a future version.
  warnings.warn(deprecation_message)


## Step 1: Find the Divergence Point

In [10]:
import torch

def find_divergence_point(
    model1,
    model2,
    prompt,
    max_new_tokens=100
):
    """
    Find the first token position where two models' generations diverge.
    """

    with model1.generate(prompt, max_new_tokens=max_new_tokens, do_sample=False):
        tokens1 = model1.generator.output.save()

    with model2.generate(prompt, max_new_tokens=max_new_tokens, do_sample=False):
        tokens2 = model2.generator.output.save()

    # find divergence point
    prompt_len = len(model1.tokenizer.tokenize(prompt))

    # compare generated tokens (after the prompt)
    for i in range(min(len(tokens1[0]) - prompt_len, len(tokens2[0]) - prompt_len)):
        if tokens1[0][prompt_len + i] != tokens2[0][prompt_len + i]:
            divergence_idx = i
            pre_div_idx = i -1

            # Decode to verify
            model1_text = model1.tokenizer.decode(tokens1[0][prompt_len:prompt_len+i])
            model2_text = model2.tokenizer.decode(tokens2[0][prompt_len:prompt_len+i])
            print(f"Models diverge at generation token {i}")
            print(f"DeepSeek: '{model1_text}'")
            print(f"Nemotron: '{model2_text}'")
            return divergence_idx, pre_div_idx, tokens1, tokens2, prompt_len, model1_text, model2_text
        
    print("Models did not diverge in the generated sequence")
    return None, None, tokens1, tokens2, prompt_len

In [43]:
prompt = "What word comes next? fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, <think>\n"

In [44]:
div_idx, pre_div_idx, deepseek_tokens, nemotron_tokens, prompt_len, deepseek_text, nemotron_text = find_divergence_point(
    deepseek_model, nemotron_model, prompt, max_new_tokens=100
)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Models diverge at generation token 5
DeepSeek: '
Okay, so I'
Nemotron: '
Okay, so I'


## Step 2: Activation Patching Across All Layers

In [54]:
def patch_thoughts(
    source_model,
    target_model, 
    prompt,
    patch_idx,
    prompt_len,
    layer_idx,
    source_model_text,
    target_model_text,
    unpatched_max_new_tokens=10000,
    patched_max_new_tokens=10000
):
    """
    Patch activations from the reasoning trace of a source_model
    into that of a target_model at a given layer and token position.
    
    Args:
        source_model: Model to extract activations from
        target_model: Model to patch activations into
        prompt: The input prompt
        patch_idx: Token position to patch (relative to start of generation)
        prompt_len: Length of the prompt in tokens
        layer_idx: Which layer to patch
        max_new_tokens: How many tokens to generate after patching
    
    Returns:
        unpatched_output: Target model's output without intervention
        patched_output: Target model's output with activation patch
    """
    # The actual sequence position is prompt_len + patch_idx
    patch_position = prompt_len + patch_idx
    
    source_thought_context = prompt + source_model_text

    target_thought_context = prompt + target_model_text
    print(f"{target_thought_context=}")
    
    # Extract activation from source model
    with source_model.trace(source_thought_context):
        # Get the activation just before the divergence point from source model
        source_activation = source_model.model.layers[layer_idx].output[0][:, patch_position, :].save()
    
    # Patch into target model and get generation
    state = {'current_line': '', 'current_line_length': 0}
    patched_tokens = []
    print(f"\nPatched Continuation:\n")
    with target_model.generate(target_thought_context, max_new_tokens=patched_max_new_tokens, do_sample=False) as tracer:
        # Patch at the divergence point
        target_model.model.layers[layer_idx].output[0][:, patch_position, :] = source_activation
        with tracer.all():

            # Access target_model output
            out = target_model.lm_head.output.save()

            # Apply softmax to obtain probs and save the result
            probs = torch.nn.functional.softmax(out, dim=-1)
            max_probs = torch.max(probs, dim=-1)
            tokens = max_probs.indices.cpu().tolist()
            patched_tokens.append(tokens[0]).save()

            state = streamer(tokens[0], target_model, max_length=160, state=state)
        patched_generation = target_model.tokenizer.decode([token[0] for token in patched_tokens]).save()
    
    # Get unpatched generation for comparison
    #state = {'current_line': '', 'current_line_length': 0}
    #unpatched_tokens = []
    #print(f"\nUnpatched Continuation:\n")
    #with target_model.generate(target_thought_context, max_new_tokens=unpatched_max_new_tokens, do_sample=False) as tracer:
    #    with tracer.all():

    #        # Access target_model output
    #        out = target_model.lm_head.output.save()

    #        # Apply softmax to obtain probs and save the result
    #        probs = torch.nn.functional.softmax(out, dim=-1)
    #        max_probs = torch.max(probs, dim=-1)
    #        tokens = max_probs.indices.cpu().tolist()
    #        unpatched_tokens.append(tokens[0]).save()

    #        state = streamer(tokens[0], target_model, max_length=160, state=state)
    #    unpatched_generation = target_model.tokenizer.decode([token[0] for token in unpatched_tokens])
    #
    return patched_generation

# Get number of layers (should be same for both models since they're both Qwen-based)
n_layers = deepseek_model.config.num_hidden_layers
print(f"Number of layers: {n_layers}")

Number of layers: 28


In [53]:
# Store results
results_deepseek_to_nemotron = {}
results_nemotron_to_deepseek = {}

"""# Patch DeepSeek → Nemotron (to fix Nemotron's loop)
print("\n" + "="*80)
print("Patching DeepSeek activations INTO Nemotron")
print("="*80)

for layer in range(n_layers):
    print(f"\nLayer {layer}/{n_layers-1}:")
    
    unpatched, patched = patch_thoughts(
        source_model=deepseek_model,  # DeepSeek (the one that doesn't loop)
        target_model=nemotron_model,  # Nemotron (the one with the loop)
        prompt=prompt,
        patch_idx=pre_div_idx,
        prompt_len=prompt_len,
        layer_idx=layer,
        source_model_text=deepseek_text,
        target_model_text=nemotron_text,
        #max_new_tokens=1000
    )
    
    results_deepseek_to_nemotron[layer] = {
        'unpatched': unpatched,
        'patched': patched
    }

    print(f"{results_deepseek_to_nemotron[layer]=}")
    
    # Show a preview
    #unpatched_text = nemotron_tokenizer.decode(unpatched[0][prompt_len:])
    #patched_text = nemotron_tokenizer.decode(patched[0][prompt_len:])
    
    # Check if they're different
    if not torch.equal(unpatched[0], patched[0]):
        print("✓ Patching changed the output")
    else:
        print("✗ Patching did not change the output")"""

# Patch Nemotron → DeepSeek (to make DeepSeek loop)
print("\n" + "="*80)
print("Patching Nemotron activations INTO DeepSeek")
print("="*80)

for layer in (11, 12, 13, 14, 15):
    print(f"\nLayer {layer}/{n_layers-1}:")
    
    patched = patch_thoughts(
        source_model=nemotron_model,  # Nemotron (the one that loops)
        target_model=deepseek_model,  # DeepSeek (the one that doesn't loop)
        prompt=prompt,
        patch_idx=pre_div_idx,
        prompt_len=prompt_len,
        layer_idx=layer,
        source_model_text=nemotron_text,
        target_model_text=deepseek_text,
        #max_new_tokens=200
    )
    
    results_nemotron_to_deepseek[layer] = {
        'patched': patched
    }
    
    # Show a preview
    # unpatched_text = deepseek_model.tokenizer.decode(unpatched[0][prompt_len:prompt_len+50])
    # patched_text = deepseek_model.tokenizer.decode(patched[0][prompt_len:prompt_len+50])
    
    #print(f"Unpatched (first 50 tokens): {unpatched_text[:200]}...")
    #print(f"Patched (first 50 tokens): {patched_text[:200]}...")


Patching Nemotron activations INTO DeepSeek

Layer 11/27:
target_thought_context='What word comes next? fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, <think>\n\nOkay, so I'


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



Patched Continuation:

 need to figure out what word comes next in this sequence: fish, card, autumn, web, fish, card, autumn, web, and then it repeats. Let me look at the pattern
 here.\n\nFirst, I notice that the sequence goes fish, card, autumn, web, and then repeats. So, the first four words are fish, card, autumn, web. Then it starts
 again with fish, card, autumn, web. So, it seems like the pattern is four words repeating over and over.\n\nLet me break it down:\n\n1. fish
2. card
3. autumn
4. web
5. fish
6. card
7. autumn
8. web
9. fish
10. card
11. autumn
12. web
13. fish
14. card
15. autumn
16. web
17. fish
18. card
19. autumn
20. web
21. fish
22. card
23. autumn
24. web
25. fish
26. card
27. autumn
28. web
29. fish
30. card
31. autumn
32. web
33. fish
34. card
35. autumn
36. web
37. fish
38. card
39. autumn
40. web
41. fish
42. card
43. autumn
44. web
45. fish
46. card
47. autumn
48. web
49. fish
50. card
51. autumn
52. web
53. fish
54. card
55. autumn
56. web
57. fish
58. ca

/disk/u/gio/.conda/envs/retrieval/lib/python3.11/site-packages/nnsight/intervention/interleaver.py:284: UserWarning: Execution complete but `model.lm_head.output.i3439` was not provided. This was in an Iterator at iteration 3439 so likely this iteration did not happen. If you were using `.iter[:]`, this is likely not an error.
  warnings.warn(msg)


UnboundLocalError: cannot access local variable 'patched_generation' where it is not associated with a value